# 🏗️ Notebook 03: 特征工程

## 学习目标
1. 理解特征工程在机器学习管道中的角色
2. 掌握时间序列的常见特征类型
3. 理解"渐进式"特征工程的理念
4. 学会用 sin/cos 编码循环特征

## 什么是特征工程？

**特征工程 = 将领域知识编码为数字，让模型能看到规律。**

一个例子:
> 你告诉 XGBoost 原始数据是 `2024-08-15 14:00 | 52000 MW`。
> XGBoost 不理解下午2点、不理解8月、不理解周四。
> 特征工程转换后: `hour=14, month=8, day_of_week=3, is_weekend=0`。
> 现在 XGBoost 可以学到: 下午2点负荷高、8月（夏天）负荷高、
> 工作日比周末负荷高。

**原始数据 vs 特征数据:**
```
timestamp            load_mw   →   hour  month  dow  weekend  lag_24h
2024-08-15 14:00     52000    →   14    8      3    0        51800
2024-08-16 14:00     52500    →   14    8      4    0        52000
2024-08-17 14:00     48000    →   14    8      5    1        52500
                ↑ 负荷更低              ↑ 周末！    ↑ 昨天值
```

## 渐进式特征工程

我们采用三层金字塔式设计:

```
        ┌─────┐
        │Tier3│ 高级: rolling统计, sin/cos编码
        ├─────┤
        │Tier2│ 中级: 节假日, 周滞后
        ├─────┤
        │Tier1│ 核心: 小时/星期/月份/周末/日滞后
        └─────┘
```

这个设计的精髓:
1. Tier 1 跑通 XGBoost → 建立 baseline
2. Tier 2 看到效果提升 → 理解增加特征=增加信息
3. Tier 3 尝试优化 → 理解特征工程的边际收益

In [ ]:
import sys
sys.path.insert(0, '..')
from pipeline.data_loader import create_loader
from pipeline.cleaner import clean_data
from pipeline.features import FeatureEngineer
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

In [ ]:
# 加载清洗后的数据
loader = create_loader('owid')
df = loader.load_data()
df = clean_data(df)

print(f"基础数据: {len(df)} 行, 列: {list(df.columns)}")

---
## Tier 1: 核心特征

5 个特征: hour, day_of_week, month, is_weekend, lag_24h

In [ ]:
engineer = FeatureEngineer()
df_t1 = engineer.add_tier1_features(df)
t1_cols = engineer.get_feature_columns('tier1')

print(f"Tier 1 特征列: {t1_cols}")
print(f"\n新 DataFrame 形状: {df_t1.shape}")
df_t1[t1_cols + ['load_mw']].head()

## Tier 2: 中级特征

+ is_holiday, lag_168h

In [ ]:
df_t2 = engineer.add_tier2_features(df_t1)
t2_cols = engineer.get_feature_columns('tier2')
print(f"Tier 2 特征列: {t2_cols}")
print(f"新增: is_holiday, lag_168h")
df_t2[t2_cols].head()

## Tier 3: 高级特征 — 循环编码

### 为什么用 sin/cos 编码小时？

如果直接把 hour 作为数值特征:
```
hour=0  (午夜)  和  hour=23 (晚上11点)  的距离是 |23-0|=23
hour=23 (晚上11点) 和 hour=0  (午夜)  的距离是 |0-23|=23
```
但午夜和晚上11点实际上是相邻的！

用 sin/cos 编码后:
```
hour_sin = sin(2π × hour / 24)
hour_cos = cos(2π × hour / 24)
```
这样 hour=23 和 hour=0 在 (sin, cos) 空间中是相邻的向量！

这也解释了为什么需要 sin AND cos —
单独的 sin 不能区分 hour=6 和 hour=18（sin 值相同但含义完全相反）
加上 cos 就能唯一确定每个小时在单位圆上的位置。

In [ ]:
df_t3 = engineer.add_tier3_features(df_t2)
t3_cols = engineer.get_feature_columns('tier3')
print(f"Tier 3 特征列 ({len(t3_cols)} 个)")

# 可视化 sin/cos 编码
import numpy as np
hours = np.arange(24)
fig = go.Figure()
fig.add_trace(go.Scatter(x=hours, y=np.sin(2*np.pi*hours/24), mode='lines+markers', name='hour_sin'))
fig.add_trace(go.Scatter(x=hours, y=np.cos(2*np.pi*hours/24), mode='lines+markers', name='hour_cos'))
fig.update_layout(title='sin/cos 循环编码', xaxis=dict(tickmode='linear', dtick=1))
fig.show()

## 📝 学习笔记

- 特征工程本质上是"告诉模型它自己看不出来的规律"
- 渐进式设计让你逐步理解每个特征的贡献
- sin/cos 编码是处理循环变量的标准方法
- **关键**: 所有 scaler 在 forecaster 内部 fold 中 fit，不在全量数据上！

**下一步: Notebook 04 → XGBoost 训练和评估**

### 思考题

1. 为什么 lag_24h 是 Tier 1 的核心特征？没有它模型会退化到什么程度？
2. hour_sin 和 hour_cos 为什么成对出现？只用一个会有什么问题？
3. 如果 Tier 3 特征加上后 MAE 反而升高了，可能是什么原因？

